# Module 7.2: 模型服务 (Model Serving)

## 1. 本章概览

### 📚 学习目标

1. **API 设计**：构建 REST API 服务模型
2. **性能优化**：批处理、缓存、负载均衡
3. **生产部署**：容器化、监控、扩展
4. **最佳实践**：可靠性、可观测性、可维护性

### 🎯 核心问题

- 如何设计高性能的模型服务？
- 如何处理大规模并发请求？
- 如何监控和优化服务性能？

### 🔑 核心概念速览

**SLA (Service Level Agreement，服务等级协议)** 是服务提供方与使用方之间关于服务质量的正式约定，通常包含延迟上限（如 P99 < 100ms）、可用性（如 99.9%）、吞吐量等指标。在本章场景中，它用于定义模型服务的性能基线，指导批处理策略、扩缩容决策和告警阈值的设定。

### ⏱️ 预计学习时间：5-6小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import matplotlib.pyplot as plt
import time
import asyncio
from collections import deque
import threading

np.random.seed(42)
torch.manual_seed(42)
print("✓ Libraries imported")

## 2. 模型服务基础

### 2.1 服务需求

**关键指标**：

1. **延迟 (Latency)**：
   - 实时服务：< 100ms
   - 批处理：< 1s
   - 离线：< 1min

2. **吞吐量 (Throughput)**：
   - 小规模：10-100 req/s
   - 中规模：100-1000 req/s
   - 大规模：1000+ req/s

3. **可靠性 (Reliability)**：
   - 可用性：99.9% ("三个九")
   - 错误率：< 0.1%
   - 恢复时间：< 5min

4. **可扩展性 (Scalability)**：
   - 水平扩展：增加实例
   - 垂直扩展：增加资源
   - 自动扩展：根据负载

### 2.2 服务模式

**同步 vs 异步**：

```
同步 (Synchronous):
  客户端 → 请求 → 服务器 → 立即响应 → 客户端
  优点: 简单，实时
  缺点: 阻塞，低吞吐

异步 (Asynchronous):
  客户端 → 请求 → 队列 → 后台处理 → 回调/轮询
  优点: 高吞吐，非阻塞
  缺点: 复杂，延迟高
```

**在线 vs 批处理**：

| 模式 | 延迟 | 吞吐量 | 适用场景 |
|------|------|--------|----------|
| 在线 (Online) | 低 | 中 | 实时推荐、聊天 |
| 批处理 (Batch) | 高 | 高 | 数据分析、ETL |
| 流式 (Streaming) | 中 | 高 | 实时监控、日志 |

### 2.3 服务架构

```
客户端
   ↓
负载均衡器 (Load Balancer)
   ↓
┌─────────┬─────────┬─────────┐
│ 服务器1  │ 服务器2  │ 服务器3  │
└─────────┴─────────┴─────────┘
   ↓         ↓         ↓
   └─────────┴─────────┘
           ↓
      模型缓存
           ↓
      监控系统
```

In [ ]:
# 🔬 Micro Practice: 简单的 Flask API

flask_example = '''
# app.py - 简单的 Flask 模型服务

from flask import Flask, request, jsonify
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

app = Flask(__name__)

# 加载模型（启动时）
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased')
model.eval()
print("Model loaded!")

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # 获取输入
        data = request.get_json()
        text = data.get('text', '')
        
        if not text:
            return jsonify({'error': 'No text provided'}), 400
        
        # 推理
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=-1).item()
            confidence = torch.softmax(logits, dim=-1).max().item()
        
        # 返回结果
        return jsonify({
            'prediction': prediction,
            'confidence': float(confidence),
            'text': text
        })
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

print("Flask API 示例:")
print("=" * 60)
print(flask_example)
print("=" * 60)

print("\n使用方法:")
print("  1. 启动服务: python app.py")
print("  2. 发送请求:")
print("     curl -X POST http://localhost:5000/predict \\")
print("          -H 'Content-Type: application/json' \\")
print("          -d '{\"text\": \"This is great!\"}'")

print("\nFlask 的问题:")
print("  ✗ 同步阻塞，低并发")
print("  ✗ 无自动文档")
print("  ✗ 无类型验证")
print("  ✗ 性能较低")

print("\n✓ Flask API 示例完成！")

In [ ]:
# 🔬 Micro Practice: 延迟分析

import matplotlib.pyplot as plt
import numpy as np

def analyze_latency():
    """
    分析不同服务模式的延迟特性
    """
    # 模拟延迟数据
    np.random.seed(42)
    
    # 同步服务：稳定但可能有排队
    sync_latency = np.random.normal(50, 10, 1000)
    sync_latency = np.clip(sync_latency, 20, 200)
    
    # 异步服务：平均延迟高但更稳定
    async_latency = np.random.normal(80, 5, 1000)
    
    # 批处理：延迟高但吞吐量大
    batch_latency = np.random.normal(150, 20, 1000)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 延迟分布
    ax1 = axes[0, 0]
    ax1.hist(sync_latency, bins=50, alpha=0.7, label='同步', color='#3498db')
    ax1.hist(async_latency, bins=50, alpha=0.7, label='异步', color='#2ecc71')
    ax1.hist(batch_latency, bins=50, alpha=0.7, label='批处理', color='#e74c3c')
    ax1.set_xlabel('延迟 (ms)', fontsize=11)
    ax1.set_ylabel('频次', fontsize=11)
    ax1.set_title('延迟分布', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 百分位数
    ax2 = axes[0, 1]
    percentiles = [50, 90, 95, 99]
    sync_p = [np.percentile(sync_latency, p) for p in percentiles]
    async_p = [np.percentile(async_latency, p) for p in percentiles]
    batch_p = [np.percentile(batch_latency, p) for p in percentiles]
    
    x = np.arange(len(percentiles))
    width = 0.25
    ax2.bar(x - width, sync_p, width, label='同步', color='#3498db')
    ax2.bar(x, async_p, width, label='异步', color='#2ecc71')
    ax2.bar(x + width, batch_p, width, label='批处理', color='#e74c3c')
    ax2.set_xlabel('百分位数', fontsize=11)
    ax2.set_ylabel('延迟 (ms)', fontsize=11)
    ax2.set_title('延迟百分位数', fontsize=12, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'P{p}' for p in percentiles])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 吞吐量对比
    ax3 = axes[1, 0]
    modes = ['同步', '异步', '批处理']
    throughput = [100, 500, 2000]  # req/s
    colors = ['#3498db', '#2ecc71', '#e74c3c']
    bars = ax3.bar(modes, throughput, color=colors, alpha=0.8)
    ax3.set_ylabel('吞吐量 (req/s)', fontsize=11)
    ax3.set_title('吞吐量对比', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=10)
    
    # 延迟 vs 吞吐量权衡
    ax4 = axes[1, 1]
    latencies = [np.mean(sync_latency), np.mean(async_latency), np.mean(batch_latency)]
    ax4.scatter(latencies, throughput, s=200, c=colors, alpha=0.6)
    
    for i, mode in enumerate(modes):
        ax4.annotate(mode, (latencies[i], throughput[i]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=10, fontweight='bold')
    
    ax4.set_xlabel('平均延迟 (ms)', fontsize=11)
    ax4.set_ylabel('吞吐量 (req/s)', fontsize=11)
    ax4.set_title('延迟 vs 吞吐量权衡', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n关键观察:")
    print("  1. 同步服务: 低延迟但吞吐量受限")
    print("  2. 异步服务: 平衡延迟和吞吐量")
    print("  3. 批处理: 高吞吐量但延迟较高")
    print("  4. 选择取决于应用场景需求")

analyze_latency()
print("\n✓ 延迟分析完成！")

## 3. FastAPI 生产服务器

### 3.1 FastAPI 优势

**为什么选择 FastAPI？**

1. **高性能**：
   - 基于 Starlette 和 Pydantic
   - 性能接近 NodeJS 和 Go
   - 支持异步 (async/await)

2. **自动文档**：
   - Swagger UI (交互式)
   - ReDoc (美观)
   - 自动生成 OpenAPI schema

3. **类型验证**：
   - 基于 Python 类型提示
   - Pydantic 模型验证
   - 自动错误处理

4. **现代特性**：
   - 依赖注入
   - 后台任务
   - WebSocket 支持

### 3.2 FastAPI vs Flask

| 特性 | Flask | FastAPI |
|------|-------|--------|
| 性能 | 中 | 高 |
| 异步支持 | 有限 | 原生 |
| 类型验证 | 手动 | 自动 |
| 文档生成 | 手动 | 自动 |
| 学习曲线 | 平缓 | 中等 |
| 生态系统 | 成熟 | 快速增长 |

In [ ]:
# 🔬 Micro Practice: FastAPI 生产服务器

fastapi_example = '''
# main.py - FastAPI 模型服务

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time
from typing import Optional

app = FastAPI(
    title="Model Serving API",
    description="Production-grade model serving with FastAPI",
    version="1.0.0"
)

# 请求模型
class PredictionRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=512, description="Input text")
    return_confidence: bool = Field(default=True, description="Return confidence score")

# 响应模型
class PredictionResponse(BaseModel):
    prediction: int
    confidence: Optional[float] = None
    latency_ms: float

# 全局模型（启动时加载）
model = None
tokenizer = None

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased')
    model.eval()
    print("Model loaded!")

@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    start_time = time.time()
    
    try:
        inputs = tokenizer(
            request.text,
            return_tensors='pt',
            truncation=True,
            max_length=512
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=-1).item()
            
            confidence = None
            if request.return_confidence:
                confidence = torch.softmax(logits, dim=-1).max().item()
        
        latency = (time.time() - start_time) * 1000
        
        return PredictionResponse(
            prediction=prediction,
            confidence=confidence,
            latency_ms=latency
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": model is not None
    }

@app.get("/metrics")
async def metrics():
    return {
        "requests_total": 0,
        "latency_avg_ms": 0,
        "error_rate": 0
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("FastAPI 生产服务器示例:")
print("=" * 60)
print(fastapi_example)
print("=" * 60)

print("\n使用方法:")
print("  1. 启动服务: uvicorn main:app --reload")
print("  2. 访问文档: http://localhost:8000/docs")
print("  3. 发送请求")

print("\nFastAPI 优势:")
print("  ✓ 自动生成交互式文档")
print("  ✓ 请求/响应自动验证")
print("  ✓ 异步支持，高并发")
print("  ✓ 类型安全")

print("\n✓ FastAPI 示例完成！")

## 4. 批处理和动态批处理

### 4.1 批处理的重要性

**为什么需要批处理？**

1. **GPU 利用率**：
   - 单个请求无法充分利用 GPU
   - 批处理可以提高 GPU 利用率 10-100×

2. **吞吐量提升**：
   $$\text{Throughput} = \frac{\text{Batch Size}}{\text{Batch Processing Time}}$$
   
   - Batch size = 1: 10 req/s
   - Batch size = 32: 200 req/s (20× 提升)

3. **成本降低**：
   - 更少的 GPU 实例
   - 更低的运营成本

### 4.2 动态批处理

**挑战**：
- 请求到达时间不确定
- 批次大小动态变化
- 需要平衡延迟和吞吐量

**策略**：

```python
while True:
    batch = []
    timeout = 50ms  # 最大等待时间
    
    # 收集请求
    while len(batch) < max_batch_size and not timeout:
        if request_available():
            batch.append(get_request())
    
    # 处理批次
    if batch:
        results = model.predict(batch)
        return_results(results)
```

### 4.3 批处理权衡

| 批次大小 | 延迟 | 吞吐量 | GPU 利用率 |
|---------|------|--------|----------|
| 1 | 10ms | 100 req/s | 10% |
| 8 | 20ms | 400 req/s | 40% |
| 32 | 50ms | 640 req/s | 80% |
| 128 | 200ms | 640 req/s | 95% |

**最优批次大小**：平衡延迟和吞吐量

In [ ]:
# 🔬 Micro Practice: 动态批处理实现

import time
import threading
from collections import deque

class DynamicBatcher:
    def __init__(self, max_batch_size=32, max_wait_ms=50):
        self.max_batch_size = max_batch_size
        self.max_wait_ms = max_wait_ms / 1000
        self.queue = deque()
        self.lock = threading.Lock()
        
    def add_request(self, request_id, data):
        with self.lock:
            self.queue.append((request_id, data, time.time()))
    
    def get_batch(self):
        batch = []
        request_ids = []
        
        with self.lock:
            start_time = time.time()
            
            while len(batch) < self.max_batch_size and self.queue:
                if time.time() - start_time > self.max_wait_ms:
                    break
                
                request_id, data, _ = self.queue.popleft()
                batch.append(data)
                request_ids.append(request_id)
        
        return batch, request_ids
    
    def process_batch(self, batch):
        time.sleep(0.01 * len(batch))
        return [f"result_{i}" for i in range(len(batch))]

# 测试
print("动态批处理测试")
print("=" * 60)

batcher = DynamicBatcher(max_batch_size=8, max_wait_ms=50)

for i in range(20):
    batcher.add_request(f"req_{i}", f"data_{i}")
    time.sleep(0.01)

batch_count = 0
total_processed = 0

while batcher.queue:
    batch, request_ids = batcher.get_batch()
    if batch:
        batch_count += 1
        results = batcher.process_batch(batch)
        total_processed += len(batch)
        print(f"批次 {batch_count}: 处理了 {len(batch)} 个请求")

print(f"\n总计:")
print(f"  批次数: {batch_count}")
print(f"  处理请求数: {total_processed}")
print(f"  平均批次大小: {total_processed / batch_count:.1f}")

print("\n✓ 动态批处理测试完成！")

In [ ]:
# 🔬 Micro Practice: 批处理性能对比

import matplotlib.pyplot as plt
import numpy as np

def benchmark_batching():
    batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
    latencies = [10, 12, 15, 20, 30, 50, 100, 200]
    throughputs = [100, 180, 300, 400, 500, 640, 700, 640]
    gpu_utils = [10, 20, 35, 50, 70, 85, 95, 98]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 延迟
    ax1 = axes[0, 0]
    ax1.plot(batch_sizes, latencies, marker='o', linewidth=2, markersize=8, color='#e74c3c')
    ax1.set_xlabel('批次大小', fontsize=11)
    ax1.set_ylabel('延迟 (ms)', fontsize=11)
    ax1.set_title('延迟 vs 批次大小', fontsize=12, fontweight='bold')
    ax1.set_xscale('log', base=2)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='延迟阈值')
    ax1.legend()
    
    # 吞吐量
    ax2 = axes[0, 1]
    ax2.plot(batch_sizes, throughputs, marker='o', linewidth=2, markersize=8, color='#2ecc71')
    ax2.set_xlabel('批次大小', fontsize=11)
    ax2.set_ylabel('吞吐量 (req/s)', fontsize=11)
    ax2.set_title('吞吐量 vs 批次大小', fontsize=12, fontweight='bold')
    ax2.set_xscale('log', base=2)
    ax2.grid(True, alpha=0.3)
    
    # GPU 利用率
    ax3 = axes[1, 0]
    ax3.plot(batch_sizes, gpu_utils, marker='o', linewidth=2, markersize=8, color='#3498db')
    ax3.set_xlabel('批次大小', fontsize=11)
    ax3.set_ylabel('GPU 利用率 (%)', fontsize=11)
    ax3.set_title('GPU 利用率 vs 批次大小', fontsize=12, fontweight='bold')
    ax3.set_xscale('log', base=2)
    ax3.set_ylim([0, 100])
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=80, color='green', linestyle='--', alpha=0.5, label='目标利用率')
    ax3.legend()
    
    # 效率
    ax4 = axes[1, 1]
    efficiency = [t / l for t, l in zip(throughputs, latencies)]
    ax4.plot(batch_sizes, efficiency, marker='o', linewidth=2, markersize=8, color='#f39c12')
    ax4.set_xlabel('批次大小', fontsize=11)
    ax4.set_ylabel('效率 (req/s/ms)', fontsize=11)
    ax4.set_title('效率 vs 批次大小', fontsize=12, fontweight='bold')
    ax4.set_xscale('log', base=2)
    ax4.grid(True, alpha=0.3)
    
    optimal_idx = np.argmax(efficiency)
    ax4.scatter([batch_sizes[optimal_idx]], [efficiency[optimal_idx]], 
               s=200, c='red', marker='*', zorder=5, label='最优点')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n最优批次大小: {batch_sizes[optimal_idx]}")
    print(f"  延迟: {latencies[optimal_idx]} ms")
    print(f"  吞吐量: {throughputs[optimal_idx]} req/s")
    print(f"  GPU 利用率: {gpu_utils[optimal_idx]}%")
    
    print("\n关键发现:")
    print("  1. 批次大小 8-32 通常是最优选择")
    print("  2. 过大的批次会导致延迟过高")
    print("  3. 需要根据 SLA 要求选择批次大小")

benchmark_batching()
print("\n✓ 批处理性能对比完成！")

## 5. 缓存和优化

### 5.1 缓存策略

**为什么需要缓存？**

1. **减少计算**：避免重复推理
2. **降低延迟**：缓存命中 < 1ms vs 推理 50ms
3. **节省成本**：减少 GPU 使用

**缓存类型**：

1. **结果缓存 (Result Caching)**：
   ```python
   cache[input_hash] = prediction
   ```
   - 适用于：相同输入重复出现
   - 命中率：10-30%（取决于应用）

2. **嵌入缓存 (Embedding Caching)**：
   ```python
   cache[text_hash] = embeddings
   ```
   - 适用于：多阶段推理
   - 节省：50-80% 计算

3. **模型缓存 (Model Caching)**：
   - 预加载模型到内存
   - 避免冷启动

### 5.2 缓存策略

**LRU (Least Recently Used)**：
- 淘汰最久未使用的项
- 实现简单，效果好

**LFU (Least Frequently Used)**：
- 淘汰使用频率最低的项
- 适合热点数据

**TTL (Time To Live)**：
- 设置过期时间
- 适合时效性数据

### 5.3 缓存性能

**命中率 (Hit Rate)**：
$$\text{Hit Rate} = \frac{\text{Cache Hits}}{\text{Total Requests}}$$

**延迟改善**：
$$\text{Avg Latency} = \text{Hit Rate} \times L_{\text{cache}} + (1 - \text{Hit Rate}) \times L_{\text{compute}}$$

**示例**：
- 缓存延迟：1ms
- 计算延迟：50ms
- 命中率：20%
- 平均延迟：1 × 0.2 + 50 × 0.8 = 40.2ms（节省 19.6%）

In [ ]:
# 🔬 Micro Practice: LRU 缓存实现

from collections import OrderedDict
import hashlib
import time

class LRUCache:
    """LRU 缓存实现"""
    def __init__(self, capacity=1000):
        self.cache = OrderedDict()
        self.capacity = capacity
        self.hits = 0
        self.misses = 0
    
    def get(self, key):
        """获取缓存"""
        if key in self.cache:
            self.hits += 1
            # 移到末尾（最近使用）
            self.cache.move_to_end(key)
            return self.cache[key]
        else:
            self.misses += 1
            return None
    
    def put(self, key, value):
        """添加到缓存"""
        if key in self.cache:
            self.cache.move_to_end(key)
        else:
            if len(self.cache) >= self.capacity:
                # 删除最久未使用的项
                self.cache.popitem(last=False)
        self.cache[key] = value
    
    def hit_rate(self):
        """计算命中率"""
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0
    
    def stats(self):
        """统计信息"""
        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': self.hit_rate(),
            'size': len(self.cache)
        }

# 测试 LRU 缓存
print("LRU 缓存测试")
print("=" * 60)

cache = LRUCache(capacity=10)

# 模拟请求
requests = ['a', 'b', 'c', 'a', 'd', 'b', 'e', 'f', 'g', 'h', 'i', 'j', 'a', 'b', 'c']

for req in requests:
    result = cache.get(req)
    if result is None:
        # 缓存未命中，模拟计算
        result = f"result_{req}"
        cache.put(req, result)
        print(f"请求 '{req}': 未命中，计算结果")
    else:
        print(f"请求 '{req}': 命中缓存")

print("\n缓存统计:")
stats = cache.stats()
for key, value in stats.items():
    if key == 'hit_rate':
        print(f"  {key}: {value:.1%}")
    else:
        print(f"  {key}: {value}")

print("\n✓ LRU 缓存测试完成！")

In [ ]:
# 🔬 Micro Practice: 缓存性能分析

import matplotlib.pyplot as plt
import numpy as np

def analyze_cache_performance():
    """
    分析缓存对性能的影响
    """
    hit_rates = np.arange(0, 1.01, 0.1)
    cache_latency = 1  # ms
    compute_latency = 50  # ms
    
    # 平均延迟
    avg_latencies = hit_rates * cache_latency + (1 - hit_rates) * compute_latency
    
    # 吞吐量提升
    baseline_throughput = 1000 / compute_latency  # req/s
    cached_throughputs = 1000 / avg_latencies
    throughput_improvement = (cached_throughputs / baseline_throughput - 1) * 100
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # 平均延迟
    ax1 = axes[0]
    ax1.plot(hit_rates * 100, avg_latencies, linewidth=2, color='#3498db')
    ax1.fill_between(hit_rates * 100, avg_latencies, alpha=0.3, color='#3498db')
    ax1.set_xlabel('缓存命中率 (%)', fontsize=11)
    ax1.set_ylabel('平均延迟 (ms)', fontsize=11)
    ax1.set_title('延迟 vs 命中率', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=compute_latency, color='red', linestyle='--', alpha=0.5, label='无缓存')
    ax1.axhline(y=cache_latency, color='green', linestyle='--', alpha=0.5, label='完全命中')
    ax1.legend()
    
    # 吞吐量提升
    ax2 = axes[1]
    ax2.plot(hit_rates * 100, throughput_improvement, linewidth=2, color='#2ecc71')
    ax2.fill_between(hit_rates * 100, 0, throughput_improvement, alpha=0.3, color='#2ecc71')
    ax2.set_xlabel('缓存命中率 (%)', fontsize=11)
    ax2.set_ylabel('吞吐量提升 (%)', fontsize=11)
    ax2.set_title('吞吐量提升 vs 命中率', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # ROI 分析
    ax3 = axes[2]
    cache_sizes = [100, 500, 1000, 5000, 10000]
    estimated_hit_rates = [0.1, 0.2, 0.3, 0.4, 0.45]  # 随缓存大小增加
    latency_reductions = [(1 - (hr * cache_latency + (1-hr) * compute_latency) / compute_latency) * 100 
                         for hr in estimated_hit_rates]
    
    ax3.plot(cache_sizes, latency_reductions, marker='o', linewidth=2, markersize=8, color='#e74c3c')
    ax3.set_xlabel('缓存大小', fontsize=11)
    ax3.set_ylabel('延迟降低 (%)', fontsize=11)
    ax3.set_title('缓存大小 vs 性能提升', fontsize=12, fontweight='bold')
    ax3.set_xscale('log')
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n关键发现:")
    print("  1. 20% 命中率可降低 19.6% 延迟")
    print("  2. 50% 命中率可降低 49% 延迟")
    print("  3. 缓存大小收益递减")
    print("  4. 需要平衡缓存大小和命中率")

analyze_cache_performance()
print("\n✓ 缓存性能分析完成！")

In [ ]:
# 🔬 Micro Practice: 请求去重

import threading
import time
from collections import defaultdict

class RequestDeduplicator:
    """请求去重器"""
    def __init__(self):
        self.pending = defaultdict(list)  # key -> [futures]
        self.lock = threading.Lock()
    
    def deduplicate(self, key, compute_fn):
        """去重并执行"""
        with self.lock:
            if key in self.pending:
                # 已有相同请求在处理中
                print(f"  请求 {key}: 去重（等待现有请求）")
                future = threading.Event()
                self.pending[key].append(future)
                return None, future
            else:
                # 新请求
                print(f"  请求 {key}: 新请求（开始处理）")
                self.pending[key] = []
                return compute_fn, None
    
    def complete(self, key, result):
        """完成请求"""
        with self.lock:
            futures = self.pending.pop(key, [])
            # 通知所有等待的请求
            for future in futures:
                future.set()
            return len(futures)

# 测试请求去重
print("请求去重测试")
print("=" * 60)

deduplicator = RequestDeduplicator()

def expensive_compute(key):
    """模拟昂贵的计算"""
    time.sleep(0.1)
    return f"result_{key}"

# 模拟并发相同请求
requests = ['A', 'A', 'B', 'A', 'C', 'B', 'A']

print("\n处理请求:")
for i, req in enumerate(requests):
    print(f"\n请求 {i+1}: {req}")
    compute_fn, future = deduplicator.deduplicate(req, lambda: expensive_compute(req))
    
    if compute_fn:
        # 执行计算
        result = compute_fn()
        duplicates = deduplicator.complete(req, result)
        print(f"  完成计算，服务了 {duplicates + 1} 个请求")
    else:
        # 等待现有请求
        print(f"  等待现有请求完成...")

print("\n统计:")
print(f"  总请求数: {len(requests)}")
print(f"  实际计算次数: {len(set(requests))}")
print(f"  节省计算: {len(requests) - len(set(requests))} 次")
print(f"  节省比例: {(1 - len(set(requests)) / len(requests)) * 100:.1f}%")

print("\n✓ 请求去重测试完成！")

## 6. 负载均衡和扩展

### 6.1 扩展策略

**垂直扩展 (Vertical Scaling)**：
- 增加单个实例的资源（CPU、内存、GPU）
- 优点：简单，无需修改架构
- 缺点：有上限，成本高

**水平扩展 (Horizontal Scaling)**：
- 增加实例数量
- 优点：无限扩展，容错性好
- 缺点：需要负载均衡，复杂度高

### 6.2 负载均衡算法

**轮询 (Round Robin)**：
```python
server_idx = (server_idx + 1) % num_servers
```
- 简单公平
- 不考虑服务器负载

**最少连接 (Least Connections)**：
```python
server = min(servers, key=lambda s: s.active_connections)
```
- 考虑当前负载
- 适合长连接

**加权轮询 (Weighted Round Robin)**：
```python
server = weighted_choice(servers, weights)
```
- 考虑服务器能力
- 适合异构环境

**一致性哈希 (Consistent Hashing)**：
```python
server = hash_ring.get_node(request_key)
```
- 缓存友好
- 适合有状态服务

### 6.3 自动扩展

**基于指标的扩展**：

$$\text{Target Replicas} = \text{Current} \times \frac{\text{Current Metric}}{\text{Target Metric}}$$

**扩展触发条件**：
- CPU 利用率 > 70%
- 内存使用 > 80%
- 请求队列长度 > 100
- 平均延迟 > 100ms

**扩展策略**：
- 快速扩容（Scale Up）：检测到负载立即扩容
- 缓慢缩容（Scale Down）：等待一段时间确认负载下降

In [ ]:
# 🔬 Micro Practice: 负载均衡器实现

import random
from collections import defaultdict

class LoadBalancer:
    """负载均衡器"""
    def __init__(self, servers, algorithm='round_robin'):
        self.servers = servers
        self.algorithm = algorithm
        self.current_idx = 0
        self.connections = defaultdict(int)
    
    def get_server(self, request_key=None):
        """选择服务器"""
        if self.algorithm == 'round_robin':
            server = self.servers[self.current_idx]
            self.current_idx = (self.current_idx + 1) % len(self.servers)
            return server
        
        elif self.algorithm == 'least_connections':
            server = min(self.servers, key=lambda s: self.connections[s])
            return server
        
        elif self.algorithm == 'random':
            return random.choice(self.servers)
        
        elif self.algorithm == 'consistent_hash':
            if request_key:
                hash_val = hash(request_key) % len(self.servers)
                return self.servers[hash_val]
            return self.servers[0]
    
    def connect(self, server):
        """记录连接"""
        self.connections[server] += 1
    
    def disconnect(self, server):
        """断开连接"""
        self.connections[server] = max(0, self.connections[server] - 1)

# 测试不同的负载均衡算法
print("负载均衡测试")
print("=" * 60)

servers = ['Server-1', 'Server-2', 'Server-3', 'Server-4']
num_requests = 20

for algorithm in ['round_robin', 'least_connections', 'random']:
    print(f"\n算法: {algorithm}")
    print("-" * 60)
    
    lb = LoadBalancer(servers, algorithm=algorithm)
    distribution = defaultdict(int)
    
    for i in range(num_requests):
        server = lb.get_server()
        lb.connect(server)
        distribution[server] += 1
        
        # 模拟一些请求完成
        if i % 5 == 0 and i > 0:
            for s in servers:
                if lb.connections[s] > 0:
                    lb.disconnect(s)
    
    print("请求分布:")
    for server in servers:
        count = distribution[server]
        percentage = count / num_requests * 100
        bar = '█' * int(percentage / 5)
        print(f"  {server}: {count:2d} ({percentage:5.1f}%) {bar}")
    
    print(f"\n当前连接数:")
    for server in servers:
        print(f"  {server}: {lb.connections[server]}")

print("\n✓ 负载均衡测试完成！")

In [ ]:
# 🔬 Micro Practice: 自动扩展模拟

import matplotlib.pyplot as plt
import numpy as np

def simulate_auto_scaling():
    """
    模拟自动扩展行为
    """
    # 模拟负载变化
    time_steps = 100
    np.random.seed(42)
    
    # 负载模式：低 -> 高峰 -> 低
    load = np.concatenate([
        np.linspace(20, 20, 20),  # 低负载
        np.linspace(20, 90, 20),  # 上升
        np.linspace(90, 90, 20),  # 高峰
        np.linspace(90, 20, 20),  # 下降
        np.linspace(20, 20, 20)   # 低负载
    ])
    load += np.random.normal(0, 5, time_steps)  # 添加噪声
    load = np.clip(load, 0, 100)
    
    # 自动扩展逻辑
    replicas = [2]  # 初始副本数
    target_cpu = 70  # 目标 CPU 利用率
    min_replicas = 2
    max_replicas = 10
    
    for i in range(1, time_steps):
        current_replicas = replicas[-1]
        current_load = load[i]
        
        # 计算每个副本的负载
        load_per_replica = current_load / current_replicas * 2  # 假设 2 个副本对应 100% 负载
        
        # 扩展决策
        if load_per_replica > target_cpu:
            # 需要扩容
            target_replicas = int(np.ceil(current_load / target_cpu * 2))
            new_replicas = min(target_replicas, current_replicas + 1, max_replicas)
        elif load_per_replica < target_cpu * 0.5:
            # 可以缩容（但要等待）
            if i > 10 and all(load[i-10:i] < target_cpu * 0.5):
                new_replicas = max(current_replicas - 1, min_replicas)
            else:
                new_replicas = current_replicas
        else:
            new_replicas = current_replicas
        
        replicas.append(new_replicas)
    
    # 可视化
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # 负载
    ax1 = axes[0]
    ax1.plot(load, linewidth=2, color='#e74c3c', label='总负载')
    ax1.axhline(y=target_cpu, color='green', linestyle='--', alpha=0.5, label='目标 CPU')
    ax1.fill_between(range(time_steps), 0, load, alpha=0.3, color='#e74c3c')
    ax1.set_ylabel('负载 (%)', fontsize=11)
    ax1.set_title('系统负载变化', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 100])
    
    # 副本数
    ax2 = axes[1]
    ax2.step(range(time_steps), replicas, where='post', linewidth=2, color='#3498db')
    ax2.fill_between(range(time_steps), min_replicas, replicas, step='post', alpha=0.3, color='#3498db')
    ax2.set_ylabel('副本数', fontsize=11)
    ax2.set_title('自动扩展响应', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, max_replicas + 1])
    
    # 每副本负载
    ax3 = axes[2]
    load_per_replica = [l / r * 2 for l, r in zip(load, replicas)]
    ax3.plot(load_per_replica, linewidth=2, color='#2ecc71', label='每副本负载')
    ax3.axhline(y=target_cpu, color='green', linestyle='--', alpha=0.5, label='目标')
    ax3.axhline(y=target_cpu * 1.2, color='red', linestyle='--', alpha=0.5, label='过载')
    ax3.fill_between(range(time_steps), 0, load_per_replica, alpha=0.3, color='#2ecc71')
    ax3.set_xlabel('时间步', fontsize=11)
    ax3.set_ylabel('负载 (%)', fontsize=11)
    ax3.set_title('每副本负载', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0, 100])
    
    plt.tight_layout()
    plt.show()
    
    print("\n自动扩展统计:")
    print(f"  最小副本数: {min(replicas)}")
    print(f"  最大副本数: {max(replicas)}")
    print(f"  平均副本数: {np.mean(replicas):.1f}")
    print(f"  扩容次数: {sum(1 for i in range(1, len(replicas)) if replicas[i] > replicas[i-1])}")
    print(f"  缩容次数: {sum(1 for i in range(1, len(replicas)) if replicas[i] < replicas[i-1])}")

simulate_auto_scaling()
print("\n✓ 自动扩展模拟完成！")

## 7. 监控和可观测性

### 7.1 可观测性三大支柱

**1. 指标 (Metrics)**：
- 数值型时间序列数据
- 用于监控和告警
- 示例：请求数、延迟、错误率

**2. 日志 (Logs)**：
- 离散事件记录
- 用于调试和审计
- 示例：请求日志、错误日志

**3. 追踪 (Traces)**：
- 请求在系统中的完整路径
- 用于性能分析
- 示例：分布式追踪

### 7.2 关键指标

**RED 方法**：

1. **Rate (速率)**：
   $$\text{RPS} = \frac{\text{Requests}}{\text{Time}}$$

2. **Errors (错误)**：
   $$\text{Error Rate} = \frac{\text{Failed Requests}}{\text{Total Requests}}$$

3. **Duration (延迟)**：
   - P50, P90, P95, P99 延迟

**USE 方法**（资源）：

1. **Utilization (利用率)**：资源使用百分比
2. **Saturation (饱和度)**：队列长度
3. **Errors (错误)**：错误计数

### 7.3 告警规则

**告警条件**：
```yaml
alert: HighErrorRate
expr: error_rate > 0.05
for: 5m
annotations:
  summary: Error rate above 5%
```

**告警级别**：
- **Critical**: 需要立即响应
- **Warning**: 需要关注
- **Info**: 仅通知

In [ ]:
# 🔬 Micro Practice: 指标收集器

import time
from collections import deque, defaultdict
import threading

class MetricsCollector:
    """指标收集器"""
    def __init__(self, window_size=60):
        self.window_size = window_size
        self.request_times = deque(maxlen=1000)
        self.latencies = deque(maxlen=1000)
        self.errors = deque(maxlen=1000)
        self.lock = threading.Lock()
    
    def record_request(self, latency_ms, is_error=False):
        """记录请求"""
        with self.lock:
            current_time = time.time()
            self.request_times.append(current_time)
            self.latencies.append(latency_ms)
            self.errors.append(1 if is_error else 0)
    
    def get_metrics(self):
        """获取指标"""
        with self.lock:
            current_time = time.time()
            cutoff_time = current_time - self.window_size
            
            # 过滤时间窗口内的数据
            recent_requests = [t for t in self.request_times if t > cutoff_time]
            recent_latencies = list(self.latencies)[-len(recent_requests):]
            recent_errors = list(self.errors)[-len(recent_requests):]
            
            if not recent_requests:
                return {
                    'rps': 0,
                    'error_rate': 0,
                    'latency_p50': 0,
                    'latency_p95': 0,
                    'latency_p99': 0
                }
            
            # 计算指标
            rps = len(recent_requests) / self.window_size
            error_rate = sum(recent_errors) / len(recent_errors)
            
            sorted_latencies = sorted(recent_latencies)
            p50_idx = int(len(sorted_latencies) * 0.50)
            p95_idx = int(len(sorted_latencies) * 0.95)
            p99_idx = int(len(sorted_latencies) * 0.99)
            
            return {
                'rps': rps,
                'error_rate': error_rate,
                'latency_p50': sorted_latencies[p50_idx] if sorted_latencies else 0,
                'latency_p95': sorted_latencies[p95_idx] if sorted_latencies else 0,
                'latency_p99': sorted_latencies[p99_idx] if sorted_latencies else 0,
                'total_requests': len(recent_requests)
            }

# 测试指标收集
print("指标收集测试")
print("=" * 60)

collector = MetricsCollector(window_size=10)

# 模拟请求
import random
for i in range(100):
    latency = random.gauss(50, 10)  # 平均 50ms
    is_error = random.random() < 0.05  # 5% 错误率
    collector.record_request(latency, is_error)
    time.sleep(0.01)

# 获取指标
metrics = collector.get_metrics()

print("\n当前指标:")
print(f"  RPS: {metrics['rps']:.2f} req/s")
print(f"  错误率: {metrics['error_rate']:.2%}")
print(f"  延迟 P50: {metrics['latency_p50']:.2f} ms")
print(f"  延迟 P95: {metrics['latency_p95']:.2f} ms")
print(f"  延迟 P99: {metrics['latency_p99']:.2f} ms")
print(f"  总请求数: {metrics['total_requests']}")

print("\n✓ 指标收集测试完成！")

In [ ]:
# 🔬 Micro Practice: 监控仪表板

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

def create_monitoring_dashboard():
    """
    创建监控仪表板
    """
    # 模拟时间序列数据
    time_points = 60
    timestamps = np.arange(time_points)
    
    # 模拟指标
    np.random.seed(42)
    rps = 100 + np.random.normal(0, 10, time_points)
    rps = np.clip(rps, 0, None)
    
    error_rate = 0.02 + np.random.normal(0, 0.01, time_points)
    error_rate = np.clip(error_rate, 0, 0.1)
    
    latency_p50 = 50 + np.random.normal(0, 5, time_points)
    latency_p95 = 80 + np.random.normal(0, 10, time_points)
    latency_p99 = 120 + np.random.normal(0, 15, time_points)
    
    cpu_util = 60 + np.random.normal(0, 10, time_points)
    cpu_util = np.clip(cpu_util, 0, 100)
    
    # 创建仪表板
    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # RPS
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(timestamps, rps, linewidth=2, color='#3498db')
    ax1.fill_between(timestamps, 0, rps, alpha=0.3, color='#3498db')
    ax1.set_title('请求速率 (RPS)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Requests/sec')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([0, time_points])
    
    # 错误率
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(timestamps, error_rate * 100, linewidth=2, color='#e74c3c')
    ax2.fill_between(timestamps, 0, error_rate * 100, alpha=0.3, color='#e74c3c')
    ax2.axhline(y=5, color='red', linestyle='--', alpha=0.5, label='告警阈值')
    ax2.set_title('错误率', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Error Rate (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim([0, time_points])
    ax2.set_ylim([0, 10])
    
    # 延迟
    ax3 = fig.add_subplot(gs[1, :])
    ax3.plot(timestamps, latency_p50, linewidth=2, label='P50', color='#2ecc71')
    ax3.plot(timestamps, latency_p95, linewidth=2, label='P95', color='#f39c12')
    ax3.plot(timestamps, latency_p99, linewidth=2, label='P99', color='#e74c3c')
    ax3.fill_between(timestamps, latency_p50, latency_p99, alpha=0.2, color='gray')
    ax3.set_title('延迟分布', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Latency (ms)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_xlim([0, time_points])
    
    # CPU 利用率
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.plot(timestamps, cpu_util, linewidth=2, color='#9b59b6')
    ax4.fill_between(timestamps, 0, cpu_util, alpha=0.3, color='#9b59b6')
    ax4.axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='警告')
    ax4.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='严重')
    ax4.set_title('CPU 利用率', fontsize=12, fontweight='bold')
    ax4.set_ylabel('CPU (%)')
    ax4.set_xlabel('Time (seconds)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim([0, time_points])
    ax4.set_ylim([0, 100])
    
    # 状态摘要
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    
    summary_text = f"""
    系统状态摘要
    ═══════════════════════════
    
    当前 RPS: {rps[-1]:.1f} req/s
    错误率: {error_rate[-1]:.2%}
    
    延迟:
      P50: {latency_p50[-1]:.1f} ms
      P95: {latency_p95[-1]:.1f} ms
      P99: {latency_p99[-1]:.1f} ms
    
    CPU 利用率: {cpu_util[-1]:.1f}%
    
    状态: {'🟢 健康' if error_rate[-1] < 0.05 and cpu_util[-1] < 80 else '🟡 警告' if error_rate[-1] < 0.1 else '🔴 严重'}
    """
    
    ax5.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
            verticalalignment='center')
    
    plt.suptitle('模型服务监控仪表板', fontsize=14, fontweight='bold')
    plt.show()
    
    print("\n监控要点:")
    print("  1. 持续监控 RPS 和错误率")
    print("  2. 关注延迟百分位数（P95, P99）")
    print("  3. 设置合理的告警阈值")
    print("  4. 定期审查和调整")

create_monitoring_dashboard()
print("\n✓ 监控仪表板创建完成！")

## 8. 容器化和部署

### 8.1 Docker 容器化

**为什么使用 Docker？**

1. **可重现性**：环境一致
2. **隔离性**：依赖隔离
3. **可移植性**：跨平台部署
4. **版本控制**：镜像版本管理

**Dockerfile 最佳实践**：

```dockerfile
# 使用官方基础镜像
FROM python:3.9-slim

# 设置工作目录
WORKDIR /app

# 复制依赖文件
COPY requirements.txt .

# 安装依赖（利用缓存）
RUN pip install --no-cache-dir -r requirements.txt

# 复制应用代码
COPY . .

# 暴露端口
EXPOSE 8000

# 健康检查
HEALTHCHECK --interval=30s --timeout=3s \
  CMD curl -f http://localhost:8000/health || exit 1

# 启动命令
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### 8.2 部署策略

**蓝绿部署 (Blue-Green)**：
```
蓝色环境（当前）  绿色环境（新版本）
      ↓                  ↓
   流量 100%          流量 0%
      ↓                  ↓
   切换流量
      ↓                  ↓
   流量 0%            流量 100%
```

**金丝雀部署 (Canary)**：
```
旧版本: 95% 流量
新版本: 5% 流量
   ↓
观察指标
   ↓
逐步增加新版本流量
```

**滚动更新 (Rolling Update)**：
```
实例 1: 更新 → 健康检查 → 完成
实例 2: 更新 → 健康检查 → 完成
实例 3: 更新 → 健康检查 → 完成
```

### 8.3 部署平台

**云平台**：
- **AWS**: ECS, EKS, Lambda
- **GCP**: Cloud Run, GKE, Cloud Functions
- **Azure**: AKS, Container Instances, Functions

**Kubernetes**：
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: model-serving
spec:
  replicas: 3
  selector:
    matchLabels:
      app: model-serving
  template:
    metadata:
      labels:
        app: model-serving
    spec:
      containers:
      - name: api
        image: model-serving:v1
        ports:
        - containerPort: 8000
        resources:
          requests:
            memory: "2Gi"
            cpu: "1"
          limits:
            memory: "4Gi"
            cpu: "2"
```

In [ ]:
# 🔬 Micro Practice: Dockerfile 示例

dockerfile_content = '''
# Dockerfile for Model Serving

# 多阶段构建
FROM python:3.9-slim as builder

WORKDIR /app

# 安装构建依赖
RUN apt-get update && apt-get install -y \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# 安装 Python 依赖
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# 最终镜像
FROM python:3.9-slim

WORKDIR /app

# 从 builder 复制依赖
COPY --from=builder /root/.local /root/.local

# 确保脚本在 PATH 中
ENV PATH=/root/.local/bin:$PATH

# 复制应用代码
COPY main.py .
COPY models/ ./models/

# 创建非 root 用户
RUN useradd -m -u 1000 appuser && \
    chown -R appuser:appuser /app
USER appuser

# 暴露端口
EXPOSE 8000

# 健康检查
HEALTHCHECK --interval=30s --timeout=3s --start-period=40s \
  CMD python -c "import requests; requests.get('http://localhost:8000/health')"

# 启动命令
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''

print("优化的 Dockerfile:")
print("=" * 60)
print(dockerfile_content)
print("=" * 60)

print("\n优化要点:")
print("  1. 多阶段构建：减小镜像大小")
print("  2. 缓存优化：先复制 requirements.txt")
print("  3. 非 root 用户：提高安全性")
print("  4. 健康检查：自动监控容器状态")
print("  5. 清理缓存：减小镜像大小")

print("\n构建和运行:")
print("  docker build -t model-serving:v1 .")
print("  docker run -p 8000:8000 model-serving:v1")

print("\n✓ Dockerfile 示例完成！")

In [ ]:
# 🔬 Micro Practice: 部署策略对比

import matplotlib.pyplot as plt
import numpy as np

def compare_deployment_strategies():
    """
    对比不同部署策略
    """
    strategies = ['蓝绿部署', '金丝雀部署', '滚动更新', 'A/B 测试']
    
    # 评分（1-10）
    risk = [3, 2, 5, 4]  # 风险（越低越好）
    speed = [9, 5, 7, 6]  # 速度（越高越好）
    complexity = [4, 7, 5, 8]  # 复杂度（越低越好）
    rollback = [10, 8, 6, 7]  # 回滚容易度（越高越好）
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
    
    # 风险
    ax1 = axes[0, 0]
    bars = ax1.barh(strategies, risk, color=colors, alpha=0.8)
    ax1.set_xlabel('风险评分 (越低越好)', fontsize=11)
    ax1.set_title('部署风险', fontsize=12, fontweight='bold')
    ax1.set_xlim([0, 10])
    ax1.invert_xaxis()  # 反转，低风险在右
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax1.text(width - 0.3, bar.get_y() + bar.get_height()/2,
                f'{risk[i]}',
                ha='right', va='center', fontsize=10, fontweight='bold')
    
    # 速度
    ax2 = axes[0, 1]
    bars = ax2.barh(strategies, speed, color=colors, alpha=0.8)
    ax2.set_xlabel('部署速度 (越高越好)', fontsize=11)
    ax2.set_title('部署速度', fontsize=12, fontweight='bold')
    ax2.set_xlim([0, 10])
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax2.text(width + 0.3, bar.get_y() + bar.get_height()/2,
                f'{speed[i]}',
                ha='left', va='center', fontsize=10, fontweight='bold')
    
    # 复杂度
    ax3 = axes[1, 0]
    bars = ax3.barh(strategies, complexity, color=colors, alpha=0.8)
    ax3.set_xlabel('实现复杂度 (越低越好)', fontsize=11)
    ax3.set_title('实现复杂度', fontsize=12, fontweight='bold')
    ax3.set_xlim([0, 10])
    ax3.invert_xaxis()
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax3.text(width - 0.3, bar.get_y() + bar.get_height()/2,
                f'{complexity[i]}',
                ha='right', va='center', fontsize=10, fontweight='bold')
    
    # 回滚容易度
    ax4 = axes[1, 1]
    bars = ax4.barh(strategies, rollback, color=colors, alpha=0.8)
    ax4.set_xlabel('回滚容易度 (越高越好)', fontsize=11)
    ax4.set_title('回滚容易度', fontsize=12, fontweight='bold')
    ax4.set_xlim([0, 10])
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax4.text(width + 0.3, bar.get_y() + bar.get_height()/2,
                f'{rollback[i]}',
                ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n部署策略选择建议:")
    print("  蓝绿部署: 快速切换，易回滚，需要双倍资源")
    print("  金丝雀部署: 风险最低，逐步验证，速度较慢")
    print("  滚动更新: 平衡方案，无需额外资源")
    print("  A/B 测试: 适合功能验证，复杂度高")

compare_deployment_strategies()
print("\n✓ 部署策略对比完成！")

## 9. 常见问题 (FAQ)

### Q1: 如何处理模型更新？

**策略**：
1. **蓝绿部署**：快速切换
2. **金丝雀部署**：逐步验证
3. **A/B 测试**：对比新旧模型

**注意事项**：
- 保持 API 兼容性
- 版本化模型
- 监控性能变化

### Q2: 如何 A/B 测试模型？

```python
def route_request(request):
    user_id = request.user_id
    if hash(user_id) % 100 < 10:  # 10% 流量
        return model_v2.predict(request)
    else:
        return model_v1.predict(request)
```

### Q3: GPU 服务如何优化？

**优化策略**：
1. **批处理**：提高 GPU 利用率
2. **模型量化**：INT8/FP16
3. **TensorRT**：优化推理
4. **多模型共享**：GPU 复用

### Q4: 如何优化冷启动？

**方法**：
1. **预热**：启动时加载模型
2. **保持活跃**：定期发送请求
3. **预留实例**：始终保持最小实例数
4. **轻量化**：减小模型大小

### Q5: 如何处理长尾延迟？

**策略**：
1. **超时控制**：设置合理超时
2. **重试机制**：失败重试
3. **降级策略**：返回缓存或默认结果
4. **异步处理**：长任务异步化

### Q6: 如何保证服务可用性？

**措施**：
1. **多副本**：至少 3 个副本
2. **健康检查**：自动重启不健康实例
3. **负载均衡**：分散流量
4. **熔断器**：防止级联失败
5. **监控告警**：及时发现问题

## 10. 总结与最佳实践

### 核心要点

1. **API 设计**：
   - 使用 FastAPI 构建生产级 API
   - Pydantic 模型验证
   - 自动文档生成

2. **性能优化**：
   - 动态批处理提高吞吐量
   - LRU 缓存减少延迟
   - 请求去重节省计算

3. **扩展性**：
   - 水平扩展应对负载
   - 负载均衡分散流量
   - 自动扩展动态调整

4. **可观测性**：
   - 收集关键指标（RPS、延迟、错误率）
   - 设置合理告警
   - 可视化监控仪表板

5. **部署**：
   - Docker 容器化
   - 选择合适的部署策略
   - Kubernetes 编排

### 最佳实践清单

**开发阶段**：
- ✅ 使用类型提示和验证
- ✅ 实现健康检查接口
- ✅ 添加指标收集
- ✅ 编写单元测试

**部署阶段**：
- ✅ 容器化应用
- ✅ 配置资源限制
- ✅ 设置健康检查
- ✅ 实现优雅关闭

**运维阶段**：
- ✅ 监控关键指标
- ✅ 设置告警规则
- ✅ 定期审查日志
- ✅ 性能调优

### 性能目标

| 指标 | 目标 | 优秀 |
|------|------|------|
| P50 延迟 | < 50ms | < 20ms |
| P99 延迟 | < 200ms | < 100ms |
| 吞吐量 | > 100 req/s | > 1000 req/s |
| 可用性 | > 99.9% | > 99.99% |
| 错误率 | < 1% | < 0.1% |

### 下一步

**Module 7.3: Model Optimization**
- 模型量化
- 知识蒸馏
- 剪枝和压缩
- 推理优化

### 💡 思考题

1. 如何在延迟和吞吐量之间找到平衡点？
2. 什么情况下应该使用同步 vs 异步 API？
3. 如何设计一个支持多模型的服务架构？
4. 如何处理模型版本管理和灰度发布？
5. 在无服务器环境中如何优化模型服务？

### 参考资源

**框架和工具**：
- [FastAPI](https://fastapi.tiangolo.com/)
- [Uvicorn](https://www.uvicorn.org/)
- [Docker](https://www.docker.com/)
- [Kubernetes](https://kubernetes.io/)
- [Prometheus](https://prometheus.io/)

**最佳实践**：
- [12-Factor App](https://12factor.net/)
- [Google SRE Book](https://sre.google/books/)
- [AWS Well-Architected](https://aws.amazon.com/architecture/well-architected/)

---

## 🎉 恭喜完成 Module 7.2！

你已经掌握了：
- ✅ 模型服务 API 设计
- ✅ 性能优化技术
- ✅ 负载均衡和扩展
- ✅ 监控和可观测性
- ✅ 容器化和部署策略
- ✅ 生产环境最佳实践

继续加油！🚀

## 思考题参考答案

### 问题 1：如何在延迟和吞吐量之间找到平衡点？

延迟与吞吐量之间存在根本性的权衡关系（Latency-Throughput Tradeoff），核心原则是**在满足 SLA 约束下最大化吞吐量**。

**关键分析框架：**

**1. 确定延迟预算（Latency Budget）**

根据业务场景制定硬性约束：
- 实时对话：P99 < 200ms
- 实时推荐：P99 < 100ms
- 离线分析：P99 < 5s

**2. 利特尔法则（Little's Law）**

$$L = \lambda \times W$$

其中 $L$ 为系统中的平均请求数，$\lambda$ 为请求到达率（吞吐量），$W$ 为平均等待时间（延迟）。

这说明在系统容量固定时，吞吐量越高，延迟越大。

**3. 动态批处理参数调优**

| 参数 | 延迟优先 | 吞吐量优先 |
|------|---------|-----------|
| max_batch_size | 4-8 | 32-128 |
| max_wait_ms | 5-10ms | 50-200ms |
| 适用场景 | 实时交互 | 批量处理 |

**4. 实践建议：**

- **分级服务**：对不同 SLA 要求的请求建立优先级队列，高优先级请求用小批次，低优先级请求用大批次聚合处理
- **自适应批处理**：根据实时流量动态调整 `max_wait_ms`，流量高时增大等待时间，流量低时减小等待时间
- **Amdahl 定律**：系统的整体加速比受限于不可并行部分，需识别真正的瓶颈（I/O、CPU、内存带宽）再优化

**通用决策流程：**
```
测量基线延迟和吞吐量
   ↓
明确 SLA 约束（P99 延迟上限）
   ↓
在满足 SLA 的前提下逐步增大批次
   ↓
观察 P99 延迟是否接近阈值
   ↓
确定最优操作点（效率曲线拐点）
```

---

### 问题 2：什么情况下应该使用同步 vs 异步 API？

**选择依据矩阵：**

| 维度 | 同步 API | 异步 API |
|------|---------|---------|
| 推理时间 | < 500ms | > 500ms 或不确定 |
| 客户端特性 | 需要立即使用结果 | 可以轮询/接受回调 |
| 并发规模 | < 100 QPS | > 100 QPS |
| 错误处理 | 简单、直接 | 需要重试机制 |
| 实现复杂度 | 低 | 高 |

**适合同步 API 的场景：**

1. **实时聊天/对话**：用户等待模型回复，延迟 < 1s 是基本要求
2. **实时搜索排序**：每次搜索需要立即返回排序结果
3. **边缘设备推理**：本地推理无网络延迟，适合同步调用
4. **简单文本分类**：推理时间稳定且短（< 100ms）

**适合异步 API 的场景：**

1. **大规模文档处理**：批量分析数千篇文档，每篇耗时 1-10s
2. **视频/图像生成**：生成任务耗时数秒至数分钟
3. **模型训练触发**：提交训练任务后异步等待结果
4. **高并发低优先级任务**：日志分析、离线推荐更新

**混合架构（推荐）：**
```
客户端
  ↓
网关层（判断请求类型）
  ├── 短任务（< 500ms）→ 同步处理 → 直接返回
  └── 长任务（> 500ms）→ 任务队列 → 返回 task_id
                                    ↓
                             客户端轮询 /status/{task_id}
                                    ↓
                             任务完成 → 返回结果
```

---

### 问题 3：如何设计一个支持多模型的服务架构？

**多模型服务面临的核心挑战：**
- 不同模型的资源需求差异大（内存、GPU、CPU）
- 模型版本更新不能影响其他模型
- 资源利用率与隔离性的矛盾

**推荐架构：模型路由 + 独立部署**

```
客户端请求
    ↓
API 网关（路由层）
    ├── 解析模型标识符（model_id / version）
    ├── 鉴权与限流
    └── 路由到对应模型服务
         ├── 模型 A 服务（情感分析）
         │     └── [3 副本] GPU 优化
         ├── 模型 B 服务（文本摘要）
         │     └── [2 副本] 大内存
         └── 模型 C 服务（分类器）
               └── [5 副本] CPU 优化
```

**关键设计决策：**

**1. 资源隔离策略**

| 方式 | 优点 | 缺点 | 适用场景 |
|------|------|------|---------|
| 独立容器（每模型一服务） | 完全隔离，独立扩缩容 | 资源利用率低 | 模型差异大、流量独立 |
| 共享进程（多模型共存） | 资源共享，利用率高 | 干扰风险，OOM 影响全部 | 小模型、资源紧张 |
| Triton Inference Server | 自动调度，GPU 共享 | 运维复杂 | 生产大规模部署 |

**2. 模型注册表（Model Registry）**

```python
# 模型注册表示例设计
model_registry = {
    "sentiment_v1": {
        "endpoint": "http://sentiment-svc:8000",
        "version": "1.0.0",
        "status": "production",
        "resources": {"memory": "2Gi", "gpu": 0}
    },
    "summarizer_v2": {
        "endpoint": "http://summarizer-svc:8001",
        "version": "2.1.0",
        "status": "production",
        "resources": {"memory": "8Gi", "gpu": 1}
    }
}
```

**3. 统一 API 接口设计**

```
POST /v1/inference
{
  "model": "sentiment",      # 模型名称
  "version": "latest",       # 版本（可选）
  "inputs": {...},           # 模型输入
  "parameters": {...}        # 推理参数
}
```

---

### 问题 4：如何处理模型版本管理和灰度发布？

**模型版本管理体系：**

**1. 语义化版本控制**

$$\text{版本号} = \text{Major.Minor.Patch}$$

- **Major**：不兼容的 API 变更（输入输出格式改变）
- **Minor**：向后兼容的功能新增（新增可选参数）
- **Patch**：性能优化、Bug 修复（行为不变）

**2. 灰度发布流程（金丝雀策略）**

```
阶段一（验证）：新版本 1% 流量
    ↓ 观察 30 分钟，指标无异常
阶段二（测试）：新版本 5% 流量
    ↓ 观察 1 小时，P99 延迟 < 阈值
阶段三（小规模）：新版本 20% 流量
    ↓ 观察 2 小时，错误率 < 0.1%
阶段四（全量）：新版本 100% 流量
    ↓ 观察 24 小时，自动回滚机制就绪
旧版本下线
```

**3. 关键监控指标对比**

在灰度期间，需要对新旧版本进行 **A/B 对比监控**：

| 指标 | 旧版本基线 | 新版本目标 | 回滚阈值 |
|------|-----------|-----------|---------|
| P99 延迟 | 150ms | ≤ 150ms | > 200ms |
| 错误率 | 0.05% | ≤ 0.05% | > 0.1% |
| 准确率（业务指标） | 85% | ≥ 85% | < 83% |

**4. 自动化回滚机制**

```python
# 伪代码：自动回滚决策
def check_canary_health(new_version_metrics, baseline_metrics):
    if new_version_metrics.error_rate > baseline_metrics.error_rate * 2:
        trigger_rollback(reason="错误率超标")
    if new_version_metrics.p99_latency > SLA_THRESHOLD:
        trigger_rollback(reason="延迟超 SLA")
    if new_version_metrics.business_metric < MINIMUM_QUALITY:
        trigger_rollback(reason="业务指标下降")
```

---

### 问题 5：在无服务器环境中如何优化模型服务？

无服务器（Serverless）环境的核心挑战是**冷启动（Cold Start）**和**资源限制**。

**冷启动问题根因分析：**

$$T_{\text{cold start}} = T_{\text{容器初始化}} + T_{\text{模型加载}} + T_{\text{预热}}$$

典型值：
- 容器初始化：500ms - 2s
- 模型加载：1s - 30s（取决于模型大小）
- 首次推理预热：100ms - 1s

**优化策略：**

**1. 减小冷启动时间**

| 方法 | 效果 | 实现复杂度 |
|------|------|-----------|
| 模型量化（INT8/FP16） | 减小加载时间 50-75% | 低 |
| 懒加载 + 分层模型 | 降低首次响应延迟 | 中 |
| 预置并发（Provisioned Concurrency） | 消除冷启动 | 低（但有成本） |
| 容器镜像优化（最小化依赖） | 减少初始化时间 | 中 |

**2. 保持"温热"状态**

```python
# 定时预热触发（每 5 分钟一次）
import boto3
import schedule

def keep_warm():
    lambda_client = boto3.client('lambda')
    lambda_client.invoke(
        FunctionName='model-inference',
        InvocationType='RequestResponse',
        Payload='{"action": "warmup"}'
    )

schedule.every(5).minutes.do(keep_warm)
```

**3. 模型分级缓存**

```
请求到来
    ↓
检查内存缓存（已加载的模型）→ 命中：直接推理（< 10ms）
    ↓ 未命中
检查文件系统缓存（/tmp 目录）→ 命中：快速加载（< 500ms）
    ↓ 未命中
从对象存储下载（S3/OSS）→ 完整加载（5-30s）
```

**4. 架构层面建议**

- **混合部署**：高频模型用常规容器（始终在线），低频模型用 Serverless
- **边缘缓存**：将热门请求的推理结果缓存在 CDN 边缘节点
- **请求合并**：用消息队列将短时间内的多个请求合并为一个 Serverless 调用，均摊冷启动成本
- **模型轻量化**：优先使用 MobileNet、DistilBERT 等为边缘设计的小模型